In [1]:
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
import time
from collections import Counter

# 1. Download Alice in Wonderland from Project Gutenberg
url = "https://www.gutenberg.org/files/11/11-0.txt"
response = urllib.request.urlopen(url)
text = response.read().decode('utf-8')

# Trim Gutenberg header/footer boilerplate if you want cleaner text
start_idx = text.find("A-CHAPTER I")
end_idx = text.find("End of Project Gutenberg's")
if start_idx != -1 and end_idx != -1:
    text = text[start_idx:end_idx]

print(f"Total dataset length: {len(text):,} characters")

# 2. Build Vocabulary (Character-level modeling is easiest and works great for Shakespeare/Alice)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size (unique characters): {vocab_size}")

# Create mapping dictionaries
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Encode entire text to integers
data = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)

# 3. Train/Validation Split (90% train, 10% val)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train tokens: {len(train_data):,} | Validation tokens: {len(val_data):,}")

Total dataset length: 144,696 characters
Vocabulary size (unique characters): 76
Train tokens: 130,226 | Validation tokens: 14,470


In [2]:
class CharacterBigramModel:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        # Initialize count matrix with 1 for Laplace smoothing to handle unseen pairs
        self.counts = torch.ones((vocab_size, vocab_size), dtype=torch.float32)
        self.transition_probs = None

    def train(self, train_tensor):
        # Count bigram transitions
        for i in range(len(train_tensor) - 1):
            c1 = train_tensor[i].item()
            c2 = train_tensor[i+1].item()
            self.counts[c1, c2] += 1.0

        # Normalize rows to get probabilities P(c2 | c1)
        self.transition_probs = self.counts / self.counts.sum(dim=1, keepdim=True)

    def compute_loss(self, val_tensor):
        # Compute Cross-Entropy Loss (Negative Log Likelihood) on validation data
        total_loss = 0.0
        n = len(val_tensor) - 1
        for i in range(n):
            c1 = val_tensor[i].item()
            c2 = val_tensor[i+1].item()
            prob = self.transition_probs[c1, c2]
            total_loss -= torch.log(prob).item()
        return total_loss / n

    def generate(self, start_char, max_length=200):
        self.current_idx = char_to_idx.get(start_char, 0)
        output = [start_char]

        for _ in range(max_length):
            probs = self.transition_probs[self.current_idx]
            # Sample next character based on probability distribution
            next_idx = torch.multinomial(probs, num_samples=1).item()
            output.append(idx_to_char[next_idx])
            self.current_idx = next_idx

        return "".join(output)

# Train and evaluate Bigram Model
start_time = time.time()
bigram_model = CharacterBigramModel(vocab_size)
bigram_model.train(train_data)
bigram_train_time = time.time() - start_time

bigram_val_loss = bigram_model.compute_loss(val_data)
bigram_sample = bigram_model.generate("A", max_length=150)

print(f"\n--- Bigram Model Results ---")
print(f"Training Time: {bigram_train_time:.4f} seconds")
print(f"Validation Loss: {bigram_val_loss:.4f}")
print(f"Generated Sample:\n{bigram_sample}")


--- Bigram Model Results ---
Training Time: 2.8108 seconds
Validation Loss: 2.4405
Generated Sample:
Ansoond HT, g d_ tivese areyou tr and t qkivwf ho tontVHoourdean

ou, mung “WQuer t tinny,” e wou s “PZeringhe shende here satiashes Mo ie, t t!”
AThta


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

seq_length = 100
batch_size = 64

def get_batches(data, seq_length, batch_size):
    # Determine number of full batches we can make
    num_batches = len(data) // (seq_length * batch_size)
    data = data[:num_batches * seq_length * batch_size]

    # Reshape into batch_size rows
    data = data.view(batch_size, -1)

    for i in range(0, data.size(1) - seq_length, seq_length):
        x = data[:, i:i+seq_length]
        y = data[:, i+1:i+seq_length+1] # Target is shifted by 1 character
        yield x, y

In [4]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2):
        super(CharLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embed = self.embedding(x)
        out, hidden = self.lstm(embed, hidden)
        out = out.contiguous().view(-1, self.hidden_dim)
        logits = self.fc(out)
        return logits, hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device),
                torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device))

# Setup device (GPU if available in Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model Hyperparameters
embedding_dim = 128
hidden_dim = 256
num_layers = 2

lstm_model = CharLSTM(vocab_size, embedding_dim, hidden_dim, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.005)

Using device: cpu


In [7]:
epochs = 3
lstm_model.train()

start_time = time.time()
for epoch in range(epochs):
    hidden = lstm_model.init_hidden(batch_size, device)
    epoch_loss = 0
    count = 0

    for x, y in get_batches(train_data, seq_length, batch_size):
        x, y = x.to(device), y.to(device)

        # Detach hidden states to prevent backpropagating through the entire history
        hidden = tuple([each.data for each in hidden])

        optimizer.zero_grad()
        logits, hidden = lstm_model(x, hidden)

        # FIX: Use .reshape(-1) instead of .view(-1)
        loss = criterion(logits, y.reshape(-1))

        loss.backward()
        # Gradient clipping to prevent exploding gradients in RNNs/LSTMs
        nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=5)
        optimizer.step()

        epoch_loss += loss.item()
        count += 1

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss/count:.4f}")

lstm_train_time = time.time() - start_time

# Validation Loss calculation
lstm_model.eval()
val_loss = 0
val_count = 0
with torch.no_grad():
    val_hidden = lstm_model.init_hidden(batch_size, device)
    for x, y in get_batches(val_data, seq_length, batch_size):
        x, y = x.to(device), y.to(device)
        logits, val_hidden = lstm_model(x, val_hidden)

        # FIX: Use .reshape(-1) here as well
        loss = criterion(logits, y.reshape(-1))

        val_loss += loss.item()
        val_count += 1

lstm_val_loss = val_loss / max(val_count, 1)
print(f"LSTM Training Time: {lstm_train_time:.2f}s | Validation Loss: {lstm_val_loss:.4f}")

Epoch 1/3 | Train Loss: 3.1704
Epoch 2/3 | Train Loss: 2.4695
Epoch 3/3 | Train Loss: 2.1421
LSTM Training Time: 53.01s | Validation Loss: 1.9865


In [8]:
def generate_lstm(model, start_str, length=200, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device
    hidden = model.init_hidden(1, device)

    input_seq = [char_to_idx[ch] for ch in start_str]
    input_tensor = torch.tensor([input_seq], dtype=torch.long, device=device)

    # Warm up hidden state with the start string
    _, hidden = model(input_tensor, hidden)

    current_idx = input_seq[-1]
    result = list(start_str)

    for _ in range(length):
        x = torch.tensor([[current_idx]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)

        # Apply temperature scaling to control randomness
        logits = logits / temperature
        probs = torch.softmax(logits, dim=-1).data

        current_idx = torch.multinomial(probs, num_samples=1).item()
        result.append(idx_to_char[current_idx])

    return "".join(result)

lstm_sample = generate_lstm(lstm_model, "The Alice", length=150)
print(f"\nGenerated Sample:\n{lstm_sample}")


Generated Sample:
The Aliced: the nelf krere
peent in in. “I mas, shits Alined her reversaind lice,” a dit.

Alide rale had the har: Afos wong, “on the kurlits etsing porsto lis


In [9]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import time

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # Causal mask to ensure model only attends to past tokens
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()

        # Query, Key, Value projections
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        # Reshape for multi-head attention: (B, num_heads, T, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v # (B, num_heads, T, head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(y)

class Block(nn.Module):
    def __init__(self, embed_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SmallTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(block_size, embed_dim)

        self.blocks = nn.Sequential(*[
            Block(embed_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.block_size, f"Cannot forward sequence of length {T}, block size is {self.block_size}"

        # Get token embeddings and positional embeddings
        tok_emb = self.token_embedding(idx) # (B, T, embed_dim)
        pos_emb = self.pos_embedding(torch.arange(T, device=idx.device)) # (T, embed_dim)

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x) # (B, T, vocab_size)

        return logits

In [10]:
block_size = 64  # Context window length
batch_size = 64
embed_dim = 128
num_heads = 4
num_layers = 2
dropout = 0.1

transformer_model = SmallTransformer(vocab_size, embed_dim, num_heads, num_layers, block_size, dropout).to(device)

def get_transformer_batch(data, block_size, batch_size):
    # Randomly sample batches from the data tensor for transformer training
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [11]:
optimizer = optim.AdamW(transformer_model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

epochs = 3
steps_per_epoch = 300
transformer_model.train()

start_time = time.time()
for epoch in range(epochs):
    epoch_loss = 0
    for step in range(steps_per_epoch):
        xb, yb = get_transformer_batch(train_data, block_size, batch_size)

        optimizer.zero_grad()
        logits = transformer_model(xb)
        # Reshape logits to match CrossEntropyLoss expectations: (B * T, vocab_size)
        loss = criterion(logits.view(-1, vocab_size), yb.view(-1))

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss/steps_per_epoch:.4f}")

transformer_train_time = time.time() - start_time

# Validation Loss calculation
transformer_model.eval()
val_loss = 0
val_steps = 50
with torch.no_grad():
    for _ in range(val_steps):
        xb, yb = get_transformer_batch(val_data, block_size, batch_size)
        logits = transformer_model(xb)
        loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
        val_loss += loss.item()

transformer_val_loss = val_loss / val_steps
print(f"Transformer Training Time: {transformer_train_time:.2f}s | Validation Loss: {transformer_val_loss:.4f}")

Epoch 1/3 | Train Loss: 2.6989
Epoch 2/3 | Train Loss: 2.2823
Epoch 3/3 | Train Loss: 2.0546
Transformer Training Time: 291.29s | Validation Loss: 1.9296


In [12]:
def generate_transformer(model, start_str, max_new_tokens=200, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device

    input_seq = [char_to_idx[ch] for ch in start_str]
    idx = torch.tensor([input_seq], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Crop context if it exceeds block_size
        idx_cond = idx[:, -block_size:]

        logits = model(idx_cond)
        # Focus only on the last time step
        logits = logits[:, -1, :] / temperature
        probs = torch.softmax(logits, dim=-1)

        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx), dim=1)

    return "".join([idx_to_char[i] for i in idx[0].tolist()])

transformer_sample = generate_transformer(transformer_model, "The Alice", max_new_tokens=150)
print(f"\nGenerated Sample:\n{transformer_sample}")


Generated Sample:
The Alice
triet.

“You’dd sel say on then, not a re dounn, the she fargeat cling orsemente bet. “I her are time
ond you’t car
the garderce, and took tred hourd
